# OMIn GraphRAG : Baselines and Comparison (Clean Version)


**Systems compared:**
1. Vanilla LLM ==> direct generation, no retrieval
2. Naive RAG ==> TF-IDF text chunk retrieval
3. KEO-style KG-RAG ==> flat weights, no guardrails (reproduces KEO failure mode)
4. Self-RAG style ==> LLM self-critique baseline
5. BYOKG-RAG simplified ==> multi-strategy graph retrieval (Mavromatis et al. 2025)
6. Semantic KG-RAG ==> cosine weighting, no guardrails (our baseline)
7. Full System == >semantic + both guardrails (our contribution)

**Metrics:**
- LLM-as-judge scores (1-5): correctness, completeness, grounding, safety, clarity, overall
- Hard unanswerable abstention by question type
- Grounding score and hallucination rate (full system only)
- Pre-guardrail schema match score distribution
- Threshold ablation
- Statistical significance tests

## 1. Imports and load all saved results (NO API calls)

In [ ]:
import json, os
import numpy as np
from collections import defaultdict, Counter
from scipy import stats

#  all pre-computed result files
with open("./output/baseline_checkpoint.json") as f:
    all_baseline_results = json.load(f)

with open("./output/eval_results_guarded.json") as f:
    guarded_results = json.load(f)

with open("./output/eval_results_baseline.json") as f:
    semantic_baseline_results = json.load(f)

with open("./output/hard_unansw_results.json") as f:
    hard_results = json.load(f)

with open("./output/llm_judge_results.json") as f:
    judge_results = json.load(f)

with open("./output/qa_benchmark_expanded.json") as f:
    benchmark = json.load(f)

# Hard unanswerable question definitions
HARD_UNANSWERABLE = benchmark.get("in_domain_unanswerable", [])

print("=== DATA LOADED ===")
print(f"Baseline systems: {list(all_baseline_results.keys())}")
print(f"Hard unanswerable questions: {len(HARD_UNANSWERABLE)}")
for sys_name, qtypes in all_baseline_results.items():
    total = sum(len(v) for v in qtypes.values())
    print(f"  {sys_name}: {total} results")
print(f"\nGuarded system results: {sum(len(v) for v in guarded_results.values())}")
print(f"Hard unanswerable results: {list(hard_results.keys())}")
judge_coverage = {s: sum(len(v) for v in qt.values())
                  for s, qt in judge_results.items()}
print(f"Judge results: {judge_coverage}")

## 2. LLM-as-Judge results — primary quality metric

In [ ]:
def get_judge_scores(sys_name, metric="overall"):
    scores = []
    for qtype in ["in_domain_answerable", "multi_hop"]:
        for q, data in judge_results.get(sys_name, {}).get(qtype, {}).items():
            val = data.get("scores", {}).get(metric)
            if val is not None:
                try:
                    scores.append(float(val))
                except:
                    pass
    return scores

METRICS = ["correctness", "completeness", "grounding", "safety", "clarity", "overall"]
SYSTEM_DISPLAY = [
    ("vanilla_llm",    "Vanilla LLM"),
    ("naive_rag",      "Naive RAG"),
    ("keo_style",      "KEO-style KG-RAG"),
    ("selfrag_style",  "Self-RAG style"),
    ("byokg_rag",      "BYOKG-RAG"),
    ("semantic_kgrag", "Semantic KG-RAG"),
    ("full_system",    "Full System (ours)"),
]

print("=== LLM-AS-JUDGE RESULTS (1-5 scale) ===")
print(f"{'System':<22}" + "".join(f"{m[:9]:>10}" for m in METRICS))
print("-" * 82)

judge_means = {}
for sys_key, sys_label in SYSTEM_DISPLAY:
    row = f"{sys_label:<22}"
    means = {}
    for m in METRICS:
        scores = get_judge_scores(sys_key, m)
        if scores:
            mean = np.mean(scores)
            means[m] = mean
            row += f"{mean:>10.2f}"
        else:
            row += f"{'N/A':>10}"
    judge_means[sys_key] = means
    print(row)

print()
print("Best system per metric:")
for m in METRICS:
    best = max(judge_means.items(), key=lambda x: x[1].get(m, 0))
    print(f"  {m}: {best[0]} ({best[1].get(m, 0):.2f})")

## 3. Statistical significance — judge scores

In [ ]:
full_scores = get_judge_scores("full_system", "overall")
print(f"Full System overall: mean={np.mean(full_scores):.3f}, n={len(full_scores)}")
print()
print(f"{'vs System':<25} {'Baseline':>10} {'Delta':>8} {'p-value':>10} {'Cohen d':>9} {'Sig':>5}")
print("-" * 70)

for sys_key, sys_label in SYSTEM_DISPLAY:
    if sys_key == "full_system":
        continue
    baseline_scores = get_judge_scores(sys_key, "overall")
    if not baseline_scores:
        continue
    t_stat, p_val = stats.ttest_ind(full_scores, baseline_scores)
    delta    = np.mean(full_scores) - np.mean(baseline_scores)
    pooled   = np.std(full_scores + baseline_scores)
    cohen_d  = delta / pooled if pooled > 0 else 0
    sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
    print(f"{sys_label:<25} {np.mean(baseline_scores):>10.3f} {delta:>+8.3f} "
          f"{p_val:>10.4f} {cohen_d:>9.3f} {sig:>5}")

## 4. Hard unanswerable abstention results

In [ ]:
print("HARD UNANSWERABLE ABSTENTION BY SYSTEM AND TYPE ")
print(f"{'System':<22} {'Total':>7} {'Spec.ID':>8} {'Aggreg.':>8} {'Missing':>8} {'FullScan':>9}")
print("-" * 65)

system_order = [
    ("vanilla_llm",   "Vanilla LLM"),
    ("naive_rag",     "Naive RAG"),
    ("keo_style",     "KEO-style"),
    ("selfrag_style", "Self-RAG style"),
    ("byokg_rag",     "BYOKG-RAG"),
    ("full_system",   "Full System (ours)"),
]

hard_abstention = {}
for sys_key, sys_label in system_order:
    results = hard_results.get(sys_key, [])
    if not results:
        print(f"{sys_label:<22} {'N/A':>7}")
        continue

    n     = len(results)
    total = sum(1 for r in results if r.get("strict_abstained", False))
    hard_abstention[sys_key] = total / n

    by_type = {}
    for r in results:
        qt = r.get("q_type_detail", "unknown")
        by_type.setdefault(qt, {"total": 0, "abstained": 0})
        by_type[qt]["total"] += 1
        if r.get("strict_abstained", False):
            by_type[qt]["abstained"] += 1

    def fmt(t):
        v = by_type.get(t, {"abstained": 0, "total": 0})
        pct = v['abstained']/v['total'] if v['total'] > 0 else 0
        return f"{pct:.0%}"

    print(f"{sys_label:<22} {total}/{n}={total/n:>4.0%}  "
          f"{fmt('specific_record_lookup'):>8} "
          f"{fmt('aggregation'):>8} "
          f"{fmt('missing_information'):>8} "
          f"{fmt('full_scan_required'):>9}")

# McNemar test: full system vs best baseline on hard unanswerable
print("\n=== STATISTICAL SIGNIFICANCE: HARD UNANSWERABLE ABSTENTION ===")
full_abs = [1 if r.get("strict_abstained") else 0
            for r in hard_results.get("full_system", [])]

for sys_key, sys_label in system_order:
    if sys_key == "full_system":
        continue
    base_abs = [1 if r.get("strict_abstained") else 0
                for r in hard_results.get(sys_key, [])]
    if not base_abs or len(base_abs) != len(full_abs):
        continue
    fs_only = sum(1 for a, b in zip(full_abs, base_abs) if a and not b)
    bl_only = sum(1 for a, b in zip(full_abs, base_abs) if not a and b)
    if fs_only + bl_only > 0:
        chi2  = (abs(fs_only - bl_only) - 1)**2 / (fs_only + bl_only)
        p_val = 1 - stats.chi2.cdf(chi2, df=1)
        sig   = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
    else:
        p_val, sig = 1.0, "ns"
    print(f"  Full System vs {sys_label:<20} "
          f"FS={np.mean(full_abs):.1%} BL={np.mean(base_abs):.1%} "
          f"p={p_val:.4f} {sig}")

## 5. Pre-guardrail score distribution and separability

In [ ]:
ans_scores  = [r.get("pre_guardrail_score") for r in
               guarded_results.get("in_domain_answerable", [])
               if r.get("pre_guardrail_score") is not None]
unan_scores = [r.get("pre_guardrail_score") for r in
               guarded_results.get("in_domain_unanswerable", [])
               if r.get("pre_guardrail_score") is not None]
mhop_scores = [r.get("pre_guardrail_score") for r in
               guarded_results.get("multi_hop", [])
               if r.get("pre_guardrail_score") is not None]

def bootstrap_ci(values, n_boot=2000, ci=0.95):
    values = np.array(values, dtype=float)
    means  = [np.mean(np.random.choice(values, size=len(values), replace=True))
              for _ in range(n_boot)]
    lo = np.percentile(means, (1-ci)/2*100)
    hi = np.percentile(means, (1+ci)/2*100)
    return np.mean(values), lo, hi

print("=== PRE-GUARDRAIL SCORE DISTRIBUTION ===")
for label, scores in [("Answerable", ans_scores),
                       ("Unanswerable", unan_scores),
                       ("Multi-hop", mhop_scores)]:
    if not scores:
        continue
    m, lo, hi = bootstrap_ci(scores)
    print(f"  {label:<15} mean={m:.3f} [{lo:.3f}, {hi:.3f}] "
          f"min={min(scores):.3f} max={max(scores):.3f} n={len(scores)}")

print()
if ans_scores and unan_scores:
    t_stat, p_val = stats.ttest_ind(ans_scores, unan_scores)
    pooled  = np.std(ans_scores + unan_scores)
    cohen_d = (np.mean(ans_scores) - np.mean(unan_scores)) / pooled
    sig     = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
    print(f"Answerable vs Unanswerable: t={t_stat:.3f}, p={p_val:.4f} {sig}, "
          f"Cohen's d={cohen_d:.3f}")
    print(f"→ This is the key result: schema match score is a "
          f"statistically significant discriminator")

## 6. Threshold ablation table

In [ ]:
def compute_threshold_metrics(guarded, threshold):
    ans_scores_all  = [(r.get("pre_guardrail_score"), r.get("abstained", False))
                       for qt in ["in_domain_answerable", "multi_hop"]
                       for r in guarded.get(qt, [])]
    unan_scores_all = [(r.get("pre_guardrail_score"), r.get("abstained", False))
                       for r in guarded.get("in_domain_unanswerable", [])]

    def abstained_at(score, already_abstained):
        if already_abstained:
            return True
        return score is not None and score < threshold

    unansw_abs = np.mean([abstained_at(s, a) for s, a in unan_scores_all]) \
                 if unan_scores_all else 0
    false_abs  = np.mean([abstained_at(s, a) for s, a in ans_scores_all]) \
                 if ans_scores_all else 0
    all_scores = [(r.get("pre_guardrail_score"), r.get("abstained", False))
                  for qt in guarded for r in guarded.get(qt, [])]
    intercept  = np.mean([abstained_at(s, a) for s, a in all_scores]) \
                 if all_scores else 0
    return unansw_abs, false_abs, intercept

print("THRESHOLD ABLATION (Full System)")
print(f"{'Threshold':<12} {'Unansw. Abstention':>20} {'False Abstention':>18} {'Interception':>14}")
print("-" * 66)

for threshold in [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]:
    u, f, i = compute_threshold_metrics(guarded_results, threshold)
    marker = " ←" if threshold == 0.70 else ""
    print(f"  {threshold:<10.2f} {u:>20.1%} {f:>18.1%} {i:>14.1%}{marker}")

print("\n← Chosen threshold (best Unanswerable Abstention / False Abstention tradeoff)")

## 7. Grounding and hallucination (full system only)

In [ ]:
all_guarded = [r for qt in guarded_results.values() for r in qt]

grounding = [r.get("post_grounding_score") for r in all_guarded
             if r.get("post_grounding_score") is not None]
halluc    = [len(r.get("hallucinated_claims", [])) for r in all_guarded]
verdicts  = Counter(r.get("post_verdict") for r in all_guarded
                    if r.get("post_verdict"))

print("=== POST-GENERATION GUARDRAIL STATS (Full System) ===")
if grounding:
    m, lo, hi = bootstrap_ci(grounding)
    print(f"  Grounding score:      {m:.3f} [{lo:.3f}, {hi:.3f}] (95% CI, n={len(grounding)})")
print(f"  Avg hallucinated claims: {np.mean(halluc):.3f}")
print(f"  Post-guardrail verdicts: {dict(verdicts)}")

# Answerable vs multi-hop grounding
for qtype in ["in_domain_answerable", "multi_hop"]:
    scores = [r.get("post_grounding_score") for r in guarded_results.get(qtype, [])
              if r.get("post_grounding_score") is not None]
    if scores:
        print(f"  {qtype}: grounding={np.mean(scores):.3f} (n={len(scores)})")

## 8. Full comparison table (paper-ready)

In [ ]:
THRESHOLD = 0.70

print("PAPER-READY COMPARISON TABLE")
print()
print(f"{'System':<22} {'Judge↑':>7} {'Grnd↑':>6} {'HardAbs↑':>9} {'OOD↑':>5} {'Halluc↓':>8} {'Records':>8}")
print("-" * 72)

for sys_key, sys_label in SYSTEM_DISPLAY:
    # LLM judge overall
    judge = get_judge_scores(sys_key, "overall")
    judge_str = f"{np.mean(judge):.2f}" if judge else "N/A"

    # LLM judge grounding dimension
    grnd = get_judge_scores(sys_key, "grounding")
    grnd_str = f"{np.mean(grnd):.2f}" if grnd else "N/A"

    # Hard unanswerable abstention
    h_results = hard_results.get(sys_key, [])
    if h_results:
        h_abs = sum(1 for r in h_results if r.get("strict_abstained")) / len(h_results)
        hard_abs_str = f"{h_abs:.1%}"
    else:
        hard_abs_str = "N/A"

    # OOD rejection
    if sys_key in all_baseline_results:
        ood = all_baseline_results[sys_key].get("out_of_domain", [])
        ood_rate = np.mean([r.get("domain_detected") == "OUT_OF_DOMAIN" for r in ood]) if ood else 0
        ood_str = f"{ood_rate:.0%}"
    elif sys_key == "semantic_kgrag":
        ood = semantic_baseline_results.get("out_of_domain", [])
        ood_rate = np.mean([r.get("domain_detected") == "OUT_OF_DOMAIN" for r in ood]) if ood else 0
        ood_str = f"{ood_rate:.0%}"
    elif sys_key == "full_system":
        ood = guarded_results.get("out_of_domain", [])
        ood_rate = np.mean([r.get("domain_detected") == "OUT_OF_DOMAIN" for r in ood]) if ood else 0
        ood_str = f"{ood_rate:.0%}"
    else:
        ood_str = "N/A"

    # Hallucination (full system only)
    if sys_key == "full_system":
        h_count = np.mean([len(r.get("hallucinated_claims", []))
                            for qt in guarded_results.values() for r in qt])
        halluc_str = f"{h_count:.2f}"
    else:
        halluc_str = "N/A"

    # Avg records retrieved
    if sys_key in all_baseline_results:
        all_r = [r for qt in all_baseline_results[sys_key].values() for r in qt]
        rec = np.mean([r.get("records_retrieved", 0) for r in all_r]) if all_r else 0
        rec_str = f"{rec:.1f}"
    else:
        rec_str = "N/A"

    print(f"{sys_label:<22} {judge_str:>7} {grnd_str:>6} {hard_abs_str:>9} "
          f"{ood_str:>5} {halluc_str:>8} {rec_str:>8}")

print()
print("↑ = higher is better, ↓ = lower is better")
print("Judge/Grnd = LLM-as-judge score (1-5 scale)")
print("HardAbs = abstention rate on hard unanswerable questions")
print("OOD = out-of-domain rejection rate")

## 9. Error analysis — qualitative examples for paper

In [ ]:
print("=== ERROR ANALYSIS ===")

# Case 1: Correct abstentions (pre-guardrail working)
print("\n--- CORRECT ABSTENTIONS (pre-guardrail caught unanswerable) ---")
full_hard = hard_results.get("full_system", [])
correct_abs = [r for r in full_hard if r.get("strict_abstained")]
for r in correct_abs[:3]:
    print(f"  Q: {r['question'][:100]}")
    print(f"  Score: {r.get('pre_guardrail_score', 'N/A'):.3f} | "
          f"Type: {r.get('q_type_detail')}")
    print(f"  Searched: {r.get('seed_nodes_searched', [])}")
    print(f"  Answer: {str(r.get('answer',''))[:150]}")
    print()

# Case 2: False abstentions (pre-guardrail too aggressive)
print("--- FALSE ABSTENTIONS (answerable but guardrail blocked) ---")
answerable_results = guarded_results.get("in_domain_answerable", [])
false_abs = [r for r in answerable_results
             if (r.get("pre_guardrail_score") or 1.0) < THRESHOLD
             and not r.get("abstained")]
for r in false_abs[:2]:
    print(f"  Q: {r['question'][:100]}")
    print(f"  Score: {r.get('pre_guardrail_score', 'N/A')}")
    print()

# Case 3: Post-guardrail catching hallucinations
print("--- POST-GUARDRAIL CATCHING HALLUCINATED CLAIMS ---")
hallucinated = [r for qt in guarded_results.values() for r in qt
                if len(r.get("hallucinated_claims", [])) > 0]
for r in hallucinated[:2]:
    print(f"  Q: {r.get('question', '')[:100]}")
    print(f"  Hallucinated: {r.get('hallucinated_claims', [])}")
    print(f"  Verdict: {r.get('post_verdict')}")
    print()

## 10. Save all final metrics for paper

In [ ]:
# gathering all paper-ready numbers in one place
paper_numbers = {
    "judge_scores": {},
    "hard_abstention": {},
    "pre_guardrail_separability": {},
    "post_guardrail": {},
    "significance": {}
}

# Judge scores
for sys_key, sys_label in SYSTEM_DISPLAY:
    scores = get_judge_scores(sys_key, "overall")
    if scores:
        m, lo, hi = bootstrap_ci(scores)
        paper_numbers["judge_scores"][sys_label] = {
            "mean": round(m, 3), "ci_low": round(lo, 3), "ci_high": round(hi, 3),
            "n": len(scores)
        }

#hard abstention
for sys_key, sys_label in system_order:
    results = hard_results.get(sys_key, [])
    if results:
        rate = sum(1 for r in results if r.get("strict_abstained")) / len(results)
        paper_numbers["hard_abstention"][sys_label] = round(rate, 3)

# Pre-guardrail separability
if ans_scores and unan_scores:
    t_stat, p_val = stats.ttest_ind(ans_scores, unan_scores)
    cohen_d = (np.mean(ans_scores) - np.mean(unan_scores)) / np.std(ans_scores + unan_scores)
    paper_numbers["pre_guardrail_separability"] = {
        "answerable_mean": round(np.mean(ans_scores), 3),
        "unanswerable_mean": round(np.mean(unan_scores), 3),
        "t_stat": round(t_stat, 3),
        "p_value": round(p_val, 4),
        "cohens_d": round(cohen_d, 3)
    }

# Post-guardrail
if grounding:
    m, lo, hi = bootstrap_ci(grounding)
    paper_numbers["post_guardrail"] = {
        "grounding_mean": round(m, 3),
        "grounding_ci": [round(lo, 3), round(hi, 3)],
        "avg_hallucinated_claims": round(np.mean(halluc), 3),
        "verdict_distribution": dict(verdicts)
    }

with open("./output/paper_numbers.json", "w") as f:
    json.dump(paper_numbers, f, indent=2)

print(json.dumps(paper_numbers, indent=2))


In [ ]:
# install locally
!pip install rouge-score bert-score --break-system-packages -q

In [ ]:
import json, numpy as np
from rouge_score import rouge_scorer

# Load all results
with open("./output/eval_results_guarded.json") as f:
    guarded = json.load(f)
with open("./output/baseline_checkpoint.json") as f:
    baselines = json.load(f)
with open("./output/eval_results_baseline.json") as f:
    semantic = json.load(f)

scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)

# For ROUGE we compare each system's answers against the full system's answers
# on the same questions (full system = pseudo-reference since it has best grounding)
full_answers = {}
for qtype in ["in_domain_answerable", "multi_hop"]:
    for r in guarded.get(qtype, []):
        if not r.get("abstained") and r.get("answer"):
            full_answers[r["question"]] = r["answer"]

systems = {
    "vanilla_llm":   baselines.get("vanilla_llm", {}),
    "naive_rag":     baselines.get("naive_rag", {}),
    "keo_style":     baselines.get("keo_style", {}),
    "selfrag_style": baselines.get("selfrag_style", {}),
    "byokg_rag":     baselines.get("byokg_rag", {}),
    "semantic_kgrag": semantic,
}

print(" ROUGE SCORES vs Full System (pseudo-reference) ")
print(f"{'System':<22} {'ROUGE-1':>9} {'ROUGE-L':>9} {'n':>5}")
print("-" * 50)

rouge_results = {}
for sys_name, results_by_type in systems.items():
    r1_scores, rl_scores = [], []
    for qtype in ["in_domain_answerable", "multi_hop"]:
        for r in results_by_type.get(qtype, []):
            q = r.get("question", "")
            if q not in full_answers:
                continue
            answer = r.get("answer", "")
            if not answer or r.get("abstained"):
                continue
            ref    = full_answers[q]
            scores = scorer.score(ref, answer)
            r1_scores.append(scores["rouge1"].fmeasure)
            rl_scores.append(scores["rougeL"].fmeasure)

    if r1_scores:
        rouge_results[sys_name] = {"rouge1": np.mean(r1_scores),
                                    "rougeL": np.mean(rl_scores),
                                    "n": len(r1_scores)}
        print(f"{sys_name:<22} {np.mean(r1_scores):>9.3f} {np.mean(rl_scores):>9.3f} {len(r1_scores):>5}")
    else:
        print(f"{sys_name:<22} {'N/A':>9} {'N/A':>9} {'0':>5}")

print("\nNote: Higher = more similar to full system answers")
print("Lower scores indicate meaningfully different answer content")

In [ ]:
from bert_score import score as bert_score_fn

print("\n=== BERTScore vs Full System ===")
print("(Loading model — first run downloads ~500MB, subsequent runs are instant)")

bert_results = {}
for sys_name, results_by_type in systems.items():
    candidates, references = [], []
    for qtype in ["in_domain_answerable", "multi_hop"]:
        for r in results_by_type.get(qtype, []):
            q = r.get("question", "")
            if q not in full_answers or not r.get("answer") or r.get("abstained"):
                continue
            candidates.append(r["answer"][:500])
            references.append(full_answers[q][:500])

    if candidates:
        P, R, F1 = bert_score_fn(candidates, references, lang="en", verbose=False)
        bert_results[sys_name] = float(F1.mean())
        print(f"{sys_name:<22} BERTScore F1: {float(F1.mean()):.4f} (n={len(candidates)})")
    else:
        print(f"{sys_name:<22} N/A")

print("\nInterpretation:")
print("  > 0.90 = very similar answers (semantic overlap)")
print("  0.80-0.90 = moderate similarity")
print("  < 0.80 = significantly different content")

with open("./output/rouge_bert_results.json", "w") as f:
    json.dump({"rouge": rouge_results, "bert": bert_results}, f, indent=2)
print("\nSaved to ./output/rouge_bert_results.json")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import json

with open("./output/eval_results_guarded.json") as f:
    guarded = json.load(f)

ans_scores  = [r["pre_guardrail_score"] for r in guarded.get("in_domain_answerable",[]) if r.get("pre_guardrail_score")]
unan_scores = [r["pre_guardrail_score"] for r in guarded.get("in_domain_unanswerable",[]) if r.get("pre_guardrail_score")]
mhop_scores = [r["pre_guardrail_score"] for r in guarded.get("multi_hop",[]) if r.get("pre_guardrail_score")]

fig, ax = plt.subplots(figsize=(8, 4))

# Box plot
data   = [ans_scores, unan_scores, mhop_scores]
labels = [f"Answerable\n(n={len(ans_scores)}, μ=0.821)",
          f"Unanswerable\n(n={len(unan_scores)}, μ=0.681)",
          f"Multi-hop\n(n={len(mhop_scores)}, μ=0.715)"]

bp = ax.boxplot(data, labels=labels, patch_artist=True,
                medianprops=dict(color="black", linewidth=2))

colors = ["#4CAF50", "#F44336", "#FF9800"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Threshold line
ax.axhline(y=0.70, color="navy", linestyle="--", linewidth=1.5, label="Threshold τ=0.70")
ax.axhspan(0, 0.70, alpha=0.05, color="red", label="Abstention zone")

ax.set_ylabel("Pre-Guardrail Schema Match Score", fontsize=11)
ax.set_title("Pre-Generation Guardrail Score Distribution by Query Type\n"
             "(t=3.072, p=0.0038**, Cohen's d=0.942)", fontsize=11)
ax.set_ylim(0.2, 1.05)
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("./output/score_distribution.pdf", dpi=300, bbox_inches="tight")
plt.savefig("./output/score_distribution.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: ./output/score_distribution.pdf and .png")